# Batch Layer - Bronze: Data Exploration & Ingestion

**Objective:** 
- Explore CSV data from S3
- Use Databricks **Auto Loader** (cloudFiles) for automatic schema detection
- Ingest raw data into Bronze Delta table
- Add metadata columns

**Key Databricks Features:**
✅ Auto Loader (cloudFiles)  
✅ Schema Detection  
✅ Streaming on Batch (availableNow)  
✅ Delta Lake ACID transactions

## 1. Environment Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, LongType, IntegerType, StringType, TimestampType
)
from pyspark.sql.functions import current_timestamp, lit, col, year, month, day
from datetime import datetime

spark = SparkSession.builder.getOrCreate()
print(f"✅ Spark Session Initialized: {spark.version}")

## 2. Define Schema & Paths

In [ ]:
# Define schema for batch data
schema = StructType([
    StructField("timestamp", LongType(), True),
    StructField("itemid", IntegerType(), True),
    StructField("property", IntegerType(), True),
    StructField("value", StringType(), True)
])

# S3 and Checkpoint paths
bronze_source_path = "s3://databricks-batch-data-demo/raw/"
checkpoint_base = "/Volumes/dev/bronze/checkpoint/_checkpoints/batch_ingestion/"
bronze_table_name = "dev.bronze.batch_item_properties"

print(f"📂 Source: {bronze_source_path}")
print(f"📂 Checkpoint: {checkpoint_base}")
print(f"📂 Target Table: {bronze_table_name}")

## 3. Sample Data Exploration (Test Read)

In [ ]:
# Quick preview of data before ingestion
sample_df = spark.read.csv(bronze_source_path, header=True, inferSchema=True)
print(f"Sample records: {sample_df.count()}")
sample_df.display()

## 4. Auto Loader Configuration (Key Databricks Feature)

In [ ]:
# Use Databricks Auto Loader for automatic file detection
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_base + "schema")
    .option("header", "true")
    .schema(schema)
    .load(bronze_source_path)
)

print("✅ Auto Loader configured")
print(f"Schema: {df.schema}")

## 5. Add Metadata Columns

In [ ]:
# Enhance with metadata
df_with_metadata = df.select(
    "*",
    current_timestamp().alias("_ingestion_time"),
    lit("s3_csv").alias("_source_file")
)

print("✅ Metadata columns added")
df_with_metadata.printSchema()

## 6. Write to Bronze Table (Streaming on Batch with availableNow)

In [ ]:
# Write to Delta table using batch triggering mode
query = (
    df_with_metadata
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_base + "data")
    .option("mergeSchema", "true")
    .trigger(availableNow=True)  # ⭐ Batch mode: process all available data once
    .toTable(bronze_table_name)
)

print(f"✅ Data ingested into {bronze_table_name}")

## 7. Validate Bronze Table

In [ ]:
# Check ingestion results
result_df = spark.sql(f"SELECT COUNT(*) as record_count FROM {bronze_table_name}")
result_df.display()

# Sample records
spark.sql(f"SELECT * FROM {bronze_table_name} LIMIT 10").display()

## 8. Data Quality Checks

In [ ]:
# Verify data completeness
spark.sql(f"""
SELECT 
  COUNT(*) as total_records,
  COUNT(timestamp) as has_timestamp,
  COUNT(itemid) as has_itemid,
  COUNT(property) as has_property,
  COUNT(value) as has_value,
  COUNT(*) - COUNT(timestamp) as missing_timestamp
FROM {bronze_table_name}
""").display()